# Section 8: Preprocessing Policy and Image Transformation Design
Defines and validates the deterministic image preprocessing pipeline that transforms raw dermoscopic images into model-ready tensors. Does not define augmentation (Section 9) or data loading mechanics (Section 10).


In [2]:
import sys
!{sys.executable} -m pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/350.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/350.9 MB 5.6 MB/s eta 0:01:03
   ---------------------------------------- 1.3/350.9 MB 3.9 MB/s eta 0:01:29
   ---------------------------------------- 1.8/350.9 MB 3.7 MB/s eta 0:01:34
   ---------------------------------------- 2.6/350.9 MB 3.7 MB/s eta 0:01:35
   ---------------------------------------- 3.4/350.9 MB 3.7 MB/s eta 0:01:36
   ---------------------------------------- 4.2/350.9 MB 3.6 MB/s eta 0:01:36
    --------------------------------------- 4.7/350.9 MB 3.6 MB/s eta 0:01:36
    --------------------------------------- 5.5/350.9 MB 3.6 MB/s eta 0:01:36
    --------------------------------------- 6.3/350.9 MB 3.5 MB/s eta 0:01:38
    --------------------------------------- 7.1/350.9 MB 3.6 MB/s eta 0:01:35
    --------------------------------------- 7.9/350.9 MB 3.6 MB/s eta 0

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 7.34.1 which is incompatible.


In [1]:
%run 01_config.ipynb

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from IPython.display import display


Creating output structure in: D:\SKIN CANCER/pipeline_output
Set basic random seeds to 42.
No GPU detected. Processing on CPU.


## Load Split Manifests & Validate Counts


In [2]:
d_splits = os.path.join(OUTPUT_ROOT, "splits")
d_preprocess = os.path.join(OUTPUT_ROOT, "audit_reports", "preprocessing")
os.makedirs(d_preprocess, exist_ok=True)

df_train = pd.read_csv(os.path.join(d_splits, "train_manifest.csv"))
df_val = pd.read_csv(os.path.join(d_splits, "val_manifest.csv"))
df_test = pd.read_csv(os.path.join(d_splits, "test_manifest.csv"))

expected = {"train": 14416, "val": 3000, "test": 3097}
expected_class = {
    "train": {"NV": 8911, "MEL": 3172, "BCC": 2333},
    "val":   {"NV": 1886, "MEL": 624,  "BCC": 490},
    "test":  {"NV": 1939, "MEL": 672,  "BCC": 486}
}

splits = {"train": df_train, "val": df_val, "test": df_test}
errors = []

print("=== SECTION 8 INPUT DATASET VALIDATION ===")
for name, df in splits.items():
    actual = len(df)
    print(f"{name}: {actual} rows (Expected: {expected[name]})")
    if actual != expected[name]:
        errors.append(f"{name} row mismatch: expected {expected[name]}, got {actual}")
    counts = df["final_authoritative_label"].value_counts()
    for cls, exp_count in expected_class[name].items():
        act_count = counts.get(cls, 0)
        if act_count != exp_count:
            errors.append(f"{name}/{cls} mismatch: expected {exp_count}, got {act_count}")

if errors:
    raise ValueError(
        "Section 8 must run on the frozen split manifests, but mismatches were found:\n- "
        + "\n- ".join(errors)
    )

print("\nAll split manifest counts match expected frozen state.")


=== SECTION 8 INPUT DATASET VALIDATION ===
train: 14416 rows (Expected: 14416)
val: 3000 rows (Expected: 3000)
test: 3097 rows (Expected: 3097)

All split manifest counts match expected frozen state.


## Define Preprocessing Policy Constants


In [3]:
# Preprocessing policy — all decisions documented in section_8_developer_ready_definitions.md
PREPROCESS_TARGET_SIZE = IMAGE_SIZE[0]  # 224 from config
PREPROCESS_INTERPOLATION = "bilinear"
PREPROCESS_COLOR_CHANNELS = 3
PREPROCESS_ASPECT_RATIO_STRATEGY = "resize_shorter_then_center_crop"
PREPROCESS_ARTIFACT_SUPPRESSION = "none_v1"

print("=== PREPROCESSING POLICY ===")
print(f"Target size: {PREPROCESS_TARGET_SIZE}x{PREPROCESS_TARGET_SIZE}")
print(f"Interpolation: {PREPROCESS_INTERPOLATION}")
print(f"Color channels: {PREPROCESS_COLOR_CHANNELS}")
print(f"Aspect ratio strategy: {PREPROCESS_ASPECT_RATIO_STRATEGY}")
print(f"Artifact suppression: {PREPROCESS_ARTIFACT_SUPPRESSION}")


=== PREPROCESSING POLICY ===
Target size: 224x224
Interpolation: bilinear
Color channels: 3
Aspect ratio strategy: resize_shorter_then_center_crop
Artifact suppression: none_v1


## Define Preprocessing Function


In [4]:
def preprocess_image(image_path, target_size=PREPROCESS_TARGET_SIZE, preprocess_fn=None):
    """
    Load, resize-and-center-crop, and normalize a single image.

    Args:
        image_path: tf.string tensor or Python string, absolute path to raw image.
        target_size: int, square output dimension (default from IMAGE_SIZE config).
        preprocess_fn: callable, backbone-specific normalization function
                       (e.g., tf.keras.applications.efficientnet.preprocess_input).
                       If None, scales pixels to [0, 1].

    Returns:
        Tensor of shape (target_size, target_size, 3), dtype float32.
    """
    # 1. Read raw bytes
    raw_bytes = tf.io.read_file(image_path)

    # 2. Decode to uint8 RGB tensor
    image = tf.image.decode_jpeg(raw_bytes, channels=3)

    # 3. Resize shorter side to target_size, preserving aspect ratio
    shape = tf.shape(image)
    h = tf.cast(shape[0], tf.float32)
    w = tf.cast(shape[1], tf.float32)
    target_f = tf.cast(target_size, tf.float32)

    # Scale factor: make the shorter side equal to target_size
    scale = target_f / tf.minimum(h, w)
    new_h = tf.cast(tf.math.ceil(h * scale), tf.int32)
    new_w = tf.cast(tf.math.ceil(w * scale), tf.int32)

    image = tf.image.resize(image, [new_h, new_w], method=PREPROCESS_INTERPOLATION)

    # 4. Center crop to (target_size, target_size)
    image = tf.image.resize_with_crop_or_pad(image, target_size, target_size)

    # 5. Cast to float32 (resize already produces float32, but be explicit)
    image = tf.cast(image, tf.float32)

    # 6. Normalize
    if preprocess_fn is not None:
        image = preprocess_fn(image)
    else:
        image = image / 255.0

    return image


def preprocess_image_default(image_path):
    """Wrapper with default normalization (scale to [0, 1]) for tf.data.map compatibility."""
    return preprocess_image(image_path, target_size=PREPROCESS_TARGET_SIZE, preprocess_fn=None)


print("Preprocessing functions defined.")


Preprocessing functions defined.


## Validation Check 1: Shape, Dtype, and Value Range


In [5]:
# Sample one image from each split to verify basic tensor properties
validation_results = []

for split_name, df in splits.items():
    sample_path = df.iloc[0]["full_path"]
    tensor = preprocess_image_default(sample_path)

    shape_ok = (tensor.shape == (PREPROCESS_TARGET_SIZE, PREPROCESS_TARGET_SIZE, 3))
    dtype_ok = (tensor.dtype == tf.float32)
    min_val = float(tf.reduce_min(tensor).numpy())
    max_val = float(tf.reduce_max(tensor).numpy())
    range_ok = (min_val >= 0.0 and max_val <= 1.0)

    print(f"{split_name}: shape={tensor.shape}, dtype={tensor.dtype}, range=[{min_val:.4f}, {max_val:.4f}]")
    print(f"  shape_ok={shape_ok}, dtype_ok={dtype_ok}, range_ok={range_ok}")

    validation_results.append({
        "check": f"{split_name}_shape", "expected": f"({PREPROCESS_TARGET_SIZE},{PREPROCESS_TARGET_SIZE},3)",
        "actual": str(tuple(tensor.shape.as_list())), "pass": shape_ok
    })
    validation_results.append({
        "check": f"{split_name}_dtype", "expected": "float32",
        "actual": str(tensor.dtype.name), "pass": dtype_ok
    })
    validation_results.append({
        "check": f"{split_name}_value_range", "expected": "[0.0, 1.0]",
        "actual": f"[{min_val:.4f}, {max_val:.4f}]", "pass": range_ok
    })

print("\nShape, dtype, and value range checks complete.")


train: shape=(224, 224, 3), dtype=<dtype: 'float32'>, range=[0.0592, 1.0000]
  shape_ok=True, dtype_ok=True, range_ok=True
val: shape=(224, 224, 3), dtype=<dtype: 'float32'>, range=[0.0461, 0.8940]
  shape_ok=True, dtype_ok=True, range_ok=True
test: shape=(224, 224, 3), dtype=<dtype: 'float32'>, range=[0.0294, 0.7548]
  shape_ok=True, dtype_ok=True, range_ok=True

Shape, dtype, and value range checks complete.


## Validation Check 2: Determinism


In [6]:
# Process the same image twice and verify identical output
determinism_path = df_train.iloc[0]["full_path"]
tensor_a = preprocess_image_default(determinism_path)
tensor_b = preprocess_image_default(determinism_path)

determinism_ok = bool(tf.reduce_all(tf.equal(tensor_a, tensor_b)).numpy())
print(f"Determinism check: same image processed twice produces identical tensors: {determinism_ok}")

validation_results.append({
    "check": "determinism", "expected": "identical",
    "actual": "identical" if determinism_ok else "different", "pass": determinism_ok
})


Determinism check: same image processed twice produces identical tensors: True


## Validation Check 3: Full Split Round-Trip Count


In [7]:
from tqdm import tqdm

def count_preprocessable(df, split_name):
    """Attempt to preprocess every image in a split manifest. Returns (success_count, failure_paths)."""
    successes = 0
    failures = []
    paths = df["full_path"].tolist()

    for path in tqdm(paths, desc=f"Preprocessing {split_name}"):
        try:
            tensor = preprocess_image_default(path)
            # Quick shape assertion as safety guard
            assert tensor.shape == (PREPROCESS_TARGET_SIZE, PREPROCESS_TARGET_SIZE, 3)
            successes += 1
        except Exception as e:
            failures.append({"path": path, "error": str(e)})

    return successes, failures


print("Running full round-trip preprocessing count on all splits...")
print("This validates that every image in every manifest can be preprocessed successfully.\n")

all_failures = []

for split_name, df in splits.items():
    successes, failures = count_preprocessable(df, split_name)
    expected_count = expected[split_name]
    count_ok = (successes == expected_count and len(failures) == 0)

    print(f"{split_name}: {successes}/{expected_count} succeeded, {len(failures)} failures. Pass: {count_ok}")

    validation_results.append({
        "check": f"{split_name}_round_trip_count", "expected": str(expected_count),
        "actual": str(successes), "pass": count_ok
    })

    all_failures.extend(failures)

if all_failures:
    print(f"\nWARNING: {len(all_failures)} preprocessing failures detected.")
    for f in all_failures[:10]:
        print(f"  {f['path']}: {f['error']}")
else:
    print("\nAll 20,513 images preprocessed successfully across all splits.")


Running full round-trip preprocessing count on all splits...
This validates that every image in every manifest can be preprocessed successfully.



Preprocessing train: 100%|██████████| 14416/14416 [01:18<00:00, 184.16it/s]


train: 14416/14416 succeeded, 0 failures. Pass: True


Preprocessing val: 100%|██████████| 3000/3000 [00:15<00:00, 187.84it/s]


val: 3000/3000 succeeded, 0 failures. Pass: True


Preprocessing test: 100%|██████████| 3097/3097 [00:16<00:00, 186.36it/s]

test: 3097/3097 succeeded, 0 failures. Pass: True

All 20,513 images preprocessed successfully across all splits.


## Validation Check 4: Visual Sanity Grids


In [8]:
def generate_preprocessing_grid(df, title, save_path):
    """Generate a 3x3 grid of preprocessed images for visual sanity checking."""
    # Sample 3 per class for a balanced grid
    sample_rows = []
    for cls in CLASS_NAMES:
        cls_df = df[df["final_authoritative_label"] == cls]
        n = min(3, len(cls_df))
        sample_rows.append(cls_df.sample(n, random_state=RANDOM_SEED))

    sample_df = pd.concat(sample_rows).reset_index(drop=True)

    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    fig.suptitle(title, fontsize=16)

    for idx, (_, row) in enumerate(sample_df.iterrows()):
        ax = axes[idx // 3][idx % 3]
        tensor = preprocess_image_default(row["full_path"])
        # Tensor is already [0, 1] so directly displayable
        ax.imshow(tensor.numpy())
        ax.set_title(f"{row['final_authoritative_label']}", fontsize=11)
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved: {save_path}")


print("Generating visual sanity grids for each split...")
generate_preprocessing_grid(
    df_train, "Preprocessed Samples: Train",
    os.path.join(d_preprocess, "preprocessing_sample_grid_train.png")
)
generate_preprocessing_grid(
    df_val, "Preprocessed Samples: Val",
    os.path.join(d_preprocess, "preprocessing_sample_grid_val.png")
)
generate_preprocessing_grid(
    df_test, "Preprocessed Samples: Test",
    os.path.join(d_preprocess, "preprocessing_sample_grid_test.png")
)

print("Visual sanity grids complete.")


Generating visual sanity grids for each split...
Saved: D:\SKIN CANCER/pipeline_output\audit_reports\preprocessing\preprocessing_sample_grid_train.png
Saved: D:\SKIN CANCER/pipeline_output\audit_reports\preprocessing\preprocessing_sample_grid_val.png
Saved: D:\SKIN CANCER/pipeline_output\audit_reports\preprocessing\preprocessing_sample_grid_test.png
Visual sanity grids complete.


## Save Preprocessing Policy & Validation Report


In [10]:
# Save preprocessing policy decisions as a CSV artifact
policy_data = [
    {"parameter": "target_size", "value": f"{PREPROCESS_TARGET_SIZE}x{PREPROCESS_TARGET_SIZE}"},
    {"parameter": "interpolation", "value": PREPROCESS_INTERPOLATION},
    {"parameter": "color_channels", "value": str(PREPROCESS_COLOR_CHANNELS)},
    {"parameter": "aspect_ratio_strategy", "value": PREPROCESS_ASPECT_RATIO_STRATEGY},
    {"parameter": "normalization_default", "value": "scale_to_0_1"},
    {"parameter": "normalization_configurable", "value": "True (preprocess_fn parameter)"},
    {"parameter": "artifact_suppression", "value": PREPROCESS_ARTIFACT_SUPPRESSION},
    {"parameter": "output_dtype", "value": "float32"},
    {"parameter": "mixed_precision_handling", "value": "Keras mixed_float16 policy handles casting internally"},
]

df_policy = pd.DataFrame(policy_data)
policy_path = os.path.join(d_preprocess, "preprocessing_policy.csv")
df_policy.to_csv(policy_path, index=False)
print(f"Preprocessing policy saved to: {policy_path}")
display(df_policy)


Preprocessing policy saved to: D:\SKIN CANCER/pipeline_output\audit_reports\preprocessing\preprocessing_policy.csv


,parameter,value
0,target_size,224x224
1,interpolation,bilinear
2,color_channels,3
3,aspect_ratio_strategy,resize_shorter_then_center_crop
4,normalization_default,scale_to_0_1
5,normalization_configurable,True (preprocess_fn parameter)
6,artifact_suppression,none_v1
7,output_dtype,float32
8,mixed_precision_handling,Keras mixed_float16 policy handles casting int...


In [11]:
# Save validation report
df_validation = pd.DataFrame(validation_results)
validation_path = os.path.join(d_preprocess, "preprocessing_validation_report.csv")
df_validation.to_csv(validation_path, index=False)
print(f"Validation report saved to: {validation_path}")
display(df_validation)

all_passed = df_validation["pass"].all()
print(f"\nAll validation checks passed: {all_passed}")

if not all_passed:
    failed = df_validation[df_validation["pass"] == False]
    print("FAILED CHECKS:")
    display(failed)


Validation report saved to: D:\SKIN CANCER/pipeline_output\audit_reports\preprocessing\preprocessing_validation_report.csv


,check,expected,actual,pass
0,train_shape,"(224,224,3)","(224, 224, 3)",True
1,train_dtype,float32,float32,True
2,train_value_range,"[0.0, 1.0]","[0.0592, 1.0000]",True
3,val_shape,"(224,224,3)","(224, 224, 3)",True
4,val_dtype,float32,float32,True
5,val_value_range,"[0.0, 1.0]","[0.0461, 0.8940]",True
6,test_shape,"(224,224,3)","(224, 224, 3)",True
7,test_dtype,float32,float32,True
8,test_value_range,"[0.0, 1.0]","[0.0294, 0.7548]",True
9,determinism,identical,identical,True



All validation checks passed: True


## Section 8 Summary


In [12]:
print("=== SECTION 8 FINAL SUMMARY ===")
print(f"Target size: {PREPROCESS_TARGET_SIZE}x{PREPROCESS_TARGET_SIZE}")
print(f"Strategy: {PREPROCESS_ASPECT_RATIO_STRATEGY}")
print(f"Interpolation: {PREPROCESS_INTERPOLATION}")
print(f"Default normalization: scale to [0, 1]")
print(f"Backbone normalization: configurable via preprocess_fn parameter")
print(f"Artifact suppression: {PREPROCESS_ARTIFACT_SUPPRESSION}")
print(f"\nTotal images validated across all splits: {sum(expected.values())}")
print(f"Preprocessing failures: {len(all_failures)}")
print(f"All validation checks passed: {all_passed}")

print("\nSaved files:")
print("- preprocessing_policy.csv")
print("- preprocessing_validation_report.csv")
print("- preprocessing_sample_grid_train.png")
print("- preprocessing_sample_grid_val.png")
print("- preprocessing_sample_grid_test.png")

print("\n=== SECTION 8 DOWNSTREAM CONTRACT ===")
print("Sections 9-12 must use preprocess_image() or preprocess_image_default() from this notebook.")
print("When a backbone is chosen in Section 11, pass its preprocess_input function as preprocess_fn.")
print("Val and test preprocessing is identical and fully deterministic.")
print("Augmentation (Section 9) is applied AFTER preprocessing, training split only.")

print("\nSection 8 completed. No raw images were modified on disk.")


=== SECTION 8 FINAL SUMMARY ===
Target size: 224x224
Strategy: resize_shorter_then_center_crop
Interpolation: bilinear
Default normalization: scale to [0, 1]
Backbone normalization: configurable via preprocess_fn parameter
Artifact suppression: none_v1

Total images validated across all splits: 20513
Preprocessing failures: 0
All validation checks passed: True

Saved files:
- preprocessing_policy.csv
- preprocessing_validation_report.csv
- preprocessing_sample_grid_train.png
- preprocessing_sample_grid_val.png
- preprocessing_sample_grid_test.png

=== SECTION 8 DOWNSTREAM CONTRACT ===
Sections 9-12 must use preprocess_image() or preprocess_image_default() from this notebook.
When a backbone is chosen in Section 11, pass its preprocess_input function as preprocess_fn.
Val and test preprocessing is identical and fully deterministic.
Augmentation (Section 9) is applied AFTER preprocessing, training split only.

Section 8 completed. No raw images were modified on disk.
